In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number."""
    return x ** 0.5


@tool
def square(x: float) -> float:
    """Calculate the square of a number."""
    return x ** 2

In [4]:
from langchain.agents import create_agent

sub_agent1 = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[square_root]
)

sub_agent2 = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[square]
)


In [5]:
from langchain.messages import HumanMessage
@tool
def call_subagent_1(x: float) -> str:
    """Call sub agent 1."""

    response = sub_agent1.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"calculate the square root of {x}"
                )
            ]
        }
    )

    return response["messages"][-1].content
@tool
def call_subagent_2(x: float) -> str:
    """Call sub agent 2."""

    response = sub_agent2.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"calculate the square of {x}"
                )
            ]
        }
    )

    return response["messages"][-1].content

In [6]:
main_agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[
        call_subagent_1,
        call_subagent_2,
    ],
    system_prompt="""
You are a helpful assistant.

Use call_subagent_1 whenever the user asks for square root.

Use call_subagent_2 whenever the user asks for square.
"""
)

In [7]:
question = "What is the square root of 456?"
response = main_agent.invoke(
    {
        "messages": [
            HumanMessage(content=question)
        ]
    }
)

In [8]:
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='e92d1450-ad64-408b-9c53-827605eca97d'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2fncxcaq3', 'function': {'arguments': '{"x":456}', 'name': 'call_subagent_1'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 313, 'total_tokens': 330, 'completion_time': 0.014226174, 'completion_tokens_details': None, 'prompt_time': 0.020364447, 'prompt_tokens_details': None, 'queue_time': 0.049163172, 'total_time': 0.034590621}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fa25a-9064-7470-be3f-4fd7f81610d9-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': '2fncxcaq3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'i